# Run image tracking on a given video input file

#### Libraries

In [ ]:
import sys

del sys.modules["src.config.settings"]

AttributeError: module 'sys' has no attribute 'module'

In [5]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from helpers import log_variable_names
from src.config.settings import (
    MODEL_PATH,
    CONFIDENCE_THRESHOLD,
    INPUT_DIR,
    INPUT_PREPROCESSED_DIR,
    OUTPUT_DIR,
    BOAT_CLASSES,
    BOTSORT_CONFIG,
)

from src.models.detector import Detector
from src.models.tracker import Tracker
from src.processors.batch_processor import BatchProcessor
from src.processors.video_preprocessor import VideoPreprocessor

log_variable_names(
    [
        "MODEL_PATH",
        "CONFIDENCE_THRESHOLD",
        "INPUT_DIR",
        "INPUT_PREPROCESSED_DIR",
        "OUTPUT_DIR",
        "BOTSORT_CONFIG",
    ],
    globals(),
)

2025-11-21 19:36:29.314 | INFO     | helpers:log_variable_names:8 - Variable 'MODEL_PATH': models/yolov8n.pt
2025-11-21 19:36:29.315 | INFO     | helpers:log_variable_names:8 - Variable 'CONFIDENCE_THRESHOLD': 0.1
2025-11-21 19:36:29.315 | INFO     | helpers:log_variable_names:8 - Variable 'INPUT_DIR': E:\01_personal_project\input
2025-11-21 19:36:29.316 | INFO     | helpers:log_variable_names:8 - Variable 'INPUT_PREPROCESSED_DIR': E:\01_personal_project\input\preprocessed
2025-11-21 19:36:29.316 | INFO     | helpers:log_variable_names:8 - Variable 'OUTPUT_DIR': E:\01_personal_project\output
2025-11-21 19:36:29.316 | INFO     | helpers:log_variable_names:8 - Variable 'BOTSORT_CONFIG': {'tracker_type': 'botsort', 'track_high_thresh': 0.25, 'track_low_thresh': 0.1, 'new_track_thresh': 0.25, 'track_buffer': 30, 'match_thresh': 0.8, 'fuse_score': True, 'gmc_method': 'sparseOptFlow', 'proximity_thresh': 0.5, 'appearance_thresh': 0.4, 'with_reid': False, 'model': 'auto'}


#### Batch Pre-Processing Video Files

In [2]:
video_preprocessor = VideoPreprocessor(
    input_dir=INPUT_DIR, output_dir=INPUT_PREPROCESSED_DIR, target_height=720
)

video_preprocessor.preprocess_all()

2025-11-20 12:48:15.552 | INFO     | src.processors.video_preprocessor:preprocess_all:132 - ============================================================
2025-11-20 12:48:15.553 | INFO     | src.processors.video_preprocessor:preprocess_all:133 - Video Preprocessing
2025-11-20 12:48:15.554 | INFO     | src.processors.video_preprocessor:preprocess_all:134 - ============================================================
2025-11-20 12:48:15.554 | INFO     | src.processors.video_preprocessor:preprocess_all:135 - Input directory:  E:\01_personal_project\input
2025-11-20 12:48:15.554 | INFO     | src.processors.video_preprocessor:preprocess_all:136 - Output directory: E:\01_personal_project\input\preprocessed
2025-11-20 12:48:15.555 | INFO     | src.processors.video_preprocessor:preprocess_all:137 - Target resolution: 720p (HD)
2025-11-20 12:48:15.555 | INFO     | src.processors.video_preprocessor:preprocess_all:138 - ============================================================
2025-11-20 12:48:

{'total': 14, 'already_processed': 0, 'copied': 0, 'resized': 14, 'skipped': 0}

## MLFLow Grid Search

In [6]:
from src.experiments.mlflow_runner import MLflowRunner
from src.config.settings import *

mlflow_runner = MLflowRunner(experiment_name="boat-tracking-grid-search")

# Run grid search (generates all combinations)
results = mlflow_runner.run_grid_search(
    config_path="../config/hyperparameter_search.yaml",
    input_dir=str(INPUT_PREPROCESSED_DIR),
    output_dir=str(OUTPUT_DIR),
    model_path=str(MODEL_PATH),
    boat_classes=BOAT_CLASSES,
)

logger.success(f"Completed {len(results)} experiments")

c:\Users\alexa\anaconda3\envs\where-the-hull-are-you\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
2025-11-21 19:36:55.498 | INFO     | src.experiments.mlflow_runner:setup_experiment:38 - MLflow experiment set: boat-tracking-grid-search
2025-11-21 19:36:55.499 | INFO     | src.experiments.mlflow_runner:setup_experiment:39 - Tracking URI: file:./mlruns
2025-11-21 19:36:55.502 | INFO     | src.experiments.mlflow_runner:run_grid_search:244 - Running grid search with 4 combinations
2025-11-21 19:36:55.502 | INFO     | src.experiments.mlflow_runner:run_grid_search:271 - 
2025-11-21 19:36:55.502 | INFO     | src.experiments.mlflow_runner:run_grid_search:272 - Grid Search Run 1/4
2025-11-21 19:36:55.503 | INFO     | src.experim

2025-11-21 19:36:57.559 | INFO     | src.models.detector:__init__:30 - Filtering for classes: ['boat', 'ship']
2025-11-21 19:36:57.559 | INFO     | src.models.detector:__init__:31 - Class IDs: [8]
2025-11-21 19:36:57.560 | INFO     | src.models.tracker:__init__:28 - Using custom tracker config with 11 parameters
2025-11-21 19:36:57.561 | INFO     | src.experiments.mlflow_runner:run_experiment:128 - Processing videos with hyperparameters:
2025-11-21 19:36:57.561 | INFO     | src.experiments.mlflow_runner:run_experiment:129 -   Confidence: 0.1
2025-11-21 19:36:57.562 | INFO     | src.experiments.mlflow_runner:run_experiment:130 -   Tracker: botsort
2025-11-21 19:36:57.562 | INFO     | src.experiments.mlflow_runner:run_experiment:131 -   BOTSORT config: {'track_high_thresh': 0.25, 'track_low_thresh': 0.1, 'new_track_thresh': 0.25, 'track_buffer': 30, 'match_thresh': 0.8, 'fuse_score': True, 'gmc_method': 'sparseOptFlow', 'proximity_thresh': 0.5, 'appearance_thresh': 0.4, 'with_reid': Fals

NameError: name 'logger' is not defined

## No MLFlow Batch Tracking

In [4]:
# Initialize with class filtering
detector = Detector(
    model_path=MODEL_PATH,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    target_classes=BOAT_CLASSES,  # Only detect boats/ships
)

tracker = Tracker(tracker_type="botsort", config=BOTSORT_CONFIG)

batch_processor = BatchProcessor(
    input_dir=INPUT_PREPROCESSED_DIR,
    output_dir=OUTPUT_DIR,
    detector=detector,
    tracker=tracker,
)

batch_processor.run()

Filtering for classes: ['boat', 'ship']
Class IDs: [8]
Using custom tracker config with 12 parameters
Found 14 video files to process

Processing 1/14: 20251026_155328.mp4
Processing 223 frames...
  Processed 30/223 frames...
  Processed 60/223 frames...
  Processed 90/223 frames...
  Processed 120/223 frames...
  Processed 150/223 frames...
  Processed 180/223 frames...
  Processed 210/223 frames...
✓ Completed: 223 frames processed
✓ Saved to: E:/01_personal_project/output\20251026_155328_processed.mp4

Processing 2/14: 20251026_154546~2.mp4
Processing 675 frames...
  Processed 30/675 frames...
  Processed 60/675 frames...
  Processed 90/675 frames...
  Processed 120/675 frames...
  Processed 150/675 frames...
  Processed 180/675 frames...
  Processed 210/675 frames...
  Processed 240/675 frames...
  Processed 270/675 frames...
  Processed 300/675 frames...
  Processed 330/675 frames...
  Processed 360/675 frames...
  Processed 390/675 frames...
  Processed 420/675 frames...
  Proces

#### Debugging and Testing

Mean detection confidence
Bounding box size statistics
Average track length
Track fragmentation rate
Frame-to-frame IoU
Processing FPS
Short track ratio (< 5 frames)
Total Processing Time